# BizNess OS — V3 Model Training Notebook
## Forward-Looking SME Survival & Profit Prediction for Cameroon

**Purpose:** Predict whether a business will survive the **next 3 years** and estimate
its projected monthly profit — anchored to real Cameroonian SME statistics.

**Model Stack:**
- Survival: LightGBM · CatBoost · Logistic Regression · Random Forest · Cox PH
- Profit: LightGBM · XGBoost · Ridge Regression
- Tuning: Optuna (Bayesian optimisation)
- Calibration: Platt Scaling / Isotonic Regression
- Explainability: SHAP

**Dataset:** 15,000 research-calibrated synthetic rows — base survival rate 17.7%
(matches MINPMEESA: 94.6% of SMEs fail within 5 years)

**Coding principles:** DRY · documented · simple variable names · no memory leaks


In [1]:
# ============================================================
# ONE-TIME INSTALL — run this cell once, then restart kernel
# ============================================================
# Uncomment the line below and run it once to install all deps.
# After it finishes, comment it back out and restart the kernel.

import subprocess, sys
pkgs = [
    "lightgbm", "catboost", "xgboost", "lifelines",
    "optuna", "shap", "scikit-learn", "pandas", "numpy",
    "matplotlib", "seaborn", "imbalanced-learn"
]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs)
print("Skip if packages already installed.")


Skip if packages already installed.


In [2]:
"""
CELL 2 — Imports and project configuration
All paths, seeds, and hyper-config live here. Change BASE_DIR if needed.
"""
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend (safe for all envs)
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- Scikit-learn ----------------------------------------------------------
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, f1_score, brier_score_loss,
    average_precision_score, mean_squared_error, r2_score,
    classification_report
)

# --- Boosting models -------------------------------------------------------
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
import xgboost as xgb

# --- Survival analysis -----------------------------------------------------
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# --- Hyperparameter optimisation -------------------------------------------
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Explainability --------------------------------------------------------
import shap

warnings.filterwarnings("ignore")

# ==========================================================================
# PROJECT PATHS
# ==========================================================================
BASE_DIR   = Path("D:/bizness_APP")           # change if different
DATA_PATH  = BASE_DIR / "BizNess_Cameroon_SME_Dataset.csv"
MODELS_DIR = BASE_DIR / "backend" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================================
# REPRODUCIBILITY & TRAINING CONFIG
# ==========================================================================
SEED          = 42
N_FOLDS       = 5     # stratified k-fold CV folds
OPTUNA_TRIALS = 60    # Bayesian optimisation trials per model
TEST_RATIO    = 0.15  # 15% final holdout
VAL_RATIO     = 0.15  # 15% validation from remaining train

np.random.seed(SEED)

print("✅ Imports OK")
print(f"📂 Dataset : {DATA_PATH}")
print(f"📂 Models  : {MODELS_DIR}")


✅ Imports OK
📂 Dataset : D:\bizness_APP\BizNess_Cameroon_SME_Dataset.csv
📂 Models  : D:\bizness_APP\backend\models


In [3]:
"""
CELL 3 — Load the research-calibrated dataset and display a quick health check.
Expected: 15,000 rows, ~17% survival rate.
"""
df_raw = pd.read_csv(DATA_PATH)

print(f"Shape            : {df_raw.shape}")
print(f"Survival rate    : {df_raw['Survived_3_Years'].mean():.1%}  (target ~17%)")
print(f"Missing values   : {df_raw.isnull().sum().sum()}")
print(f"\nColumns ({len(df_raw.columns)}):")
print(df_raw.dtypes.to_string())
df_raw.head(3)


Shape            : (15000, 44)
Survival rate    : 17.7%  (target ~17%)
Missing values   : 0

Columns (44):
Business_ID                         int64
Region                             object
Location                           object
Urban_Rural                        object
Sector                             object
Business_Type                      object
SME_Category_Cameroon              object
Startup_Capital_CFA                 int64
Employees                           int64
Monthly_Profit_CFA                  int64
Market_Competition                 object
Business_Age_Years                float64
Years_of_Experience               float64
Owner_Education                    object
Access_to_Financing                object
Owner_Age                           int64
Owner_Gender                       object
Registered_Formal                  object
Tax_Compliance_Level               object
Primary_Constraint                 object
Secondary_Constraint               object
Informal_Co

,Business_ID,Region,Location,Urban_Rural,Sector,Business_Type,SME_Category_Cameroon,Startup_Capital_CFA,Employees,Monthly_Profit_CFA,...,Has_Business_Plan,Formal_Financial_Records,Survival_Probability_Simulated,Risk_Level_Simulated,Survived_3_Years,total_overhead_pct,capital_per_employee,risk_per_experience,infra_energy_pressure,profit_on_capital
0,1,Centre,Esse,Urban,Retail & Wholesale Trade,Trade,Small,866264,8,7335,...,No,No,0.1243,High,0,28.53,108283.00,0.0171,2.2418,0.0085
1,2,Littoral,Nkongsamba,Urban,Retail & Wholesale Trade,Trade,Micro,204361,1,5000,...,No,Yes,0.0200,High,0,18.69,204361.00,0.0774,1.9440,0.0245
2,3,West,Bafang,Rural,Retail & Wholesale Trade,Trade,Micro,567215,3,5000,...,No,No,0.0339,High,0,18.58,189071.67,0.0578,3.6920,0.0088


In [4]:
"""
CELL 4 — Add two new features identified from research documents:

1. Power_Outage_Frequency  (World Bank 2024: 93.3% affected, avg 10.4/month)
   Derived from Region_Electricity_Index so it is correlated with survival.

2. Financing_Source  (MINPMEESA 2024: categorical financing type)
   Derived from Access_to_Financing to preserve the existing survival signal.
"""
import copy

def add_power_outage_frequency(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """
    Compute outage frequency per firm.
    Lower electricity index → more outages.
    Formula: base = 25 - 22 * elec_idx  (gives ~17/mo in Far North, ~6/mo in Littoral)
    93.3 % of firms experience at least one outage per month.
    """
    rng = np.random.default_rng(seed)

    base_outages = 25.0 - 22.0 * df["Region_Electricity_Index"].values
    noisy_outages = rng.normal(base_outages, 2.5)
    counts = np.maximum(0, noisy_outages).astype(int)

    # 6.7 % of firms have zero outages
    has_outage = rng.random(len(df)) < 0.933
    df = df.copy()
    df["Power_Outage_Frequency"] = np.where(has_outage, counts, 0)
    return df


def add_financing_source(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """
    Assign a categorical financing source consistent with Access_to_Financing.

    Access_to_Financing == 'Yes'  →  Bank Loan | Government Subsidy | Supplier Credit
    Access_to_Financing == 'No'   →  Own Resources | Tontine

    Probabilities anchored to MINPMEESA 2024 financing structure survey.
    """
    rng = np.random.default_rng(seed)
    df = df.copy()

    formal_sources    = ["Bank Loan", "Government Subsidy", "Supplier Credit"]
    formal_probs      = [0.45, 0.35, 0.20]
    informal_sources  = ["Own Resources", "Tontine"]
    informal_probs    = [0.88, 0.12]

    has_financing = df["Access_to_Financing"].values == "Yes"

    formal_choices   = rng.choice(formal_sources, p=formal_probs, size=len(df))
    informal_choices = rng.choice(informal_sources, p=informal_probs, size=len(df))

    df["Financing_Source"] = np.where(has_financing, formal_choices, informal_choices)
    return df


# Apply both enrichment functions
df = df_raw.pipe(add_power_outage_frequency, seed=SEED).pipe(add_financing_source, seed=SEED)

print("New columns added:")
print(f"  Power_Outage_Frequency  — mean {df['Power_Outage_Frequency'].mean():.1f}/mo, "
      f"zero={( df['Power_Outage_Frequency']==0).mean():.1%}")
print(f"  Financing_Source        — {df['Financing_Source'].value_counts().to_dict()}")
df[["Region", "Region_Electricity_Index", "Power_Outage_Frequency", "Access_to_Financing", "Financing_Source"]].head(5)


New columns added:
  Power_Outage_Frequency  — mean 8.6/mo, zero=7.0%
  Financing_Source        — {'Own Resources': 9062, 'Bank Loan': 2102, 'Government Subsidy': 1689, 'Tontine': 1254, 'Supplier Credit': 893}


,Region,Region_Electricity_Index,Power_Outage_Frequency,Access_to_Financing,Financing_Source
0,Centre,0.75,9,No,Own Resources
1,Littoral,0.81,4,No,Own Resources
2,West,0.70,0,No,Own Resources
3,Centre,0.76,10,No,Own Resources
4,West,0.73,4,Yes,Bank Loan


In [5]:
"""
CELL 5 — Exploratory Data Analysis
Key charts: class balance, survival by region/sector/education, correlations.
"""
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("BizNess OS V3 — Exploratory Data Analysis", fontsize=14, fontweight="bold")

# 1. Class balance
ax = axes[0, 0]
counts = df["Survived_3_Years"].value_counts().sort_index()
ax.bar(["Failed (0)", "Survived (1)"], counts.values,
       color=["#e74c3c", "#2ecc71"], edgecolor="white")
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f"{v:,}\n({v/len(df):.1%})", ha="center", fontsize=9)

# 2. Survival rate by region
ax = axes[0, 1]
surv_region = (df.groupby("Region")["Survived_3_Years"]
               .mean().sort_values(ascending=True))
surv_region.plot(kind="barh", ax=ax, color="#3498db")
ax.set_title("Survival Rate by Region")
ax.set_xlabel("Survival Rate")
ax.axvline(0.177, color="red", linestyle="--", alpha=0.7, label="Avg 17.7%")
ax.legend(fontsize=8)

# 3. Survival rate by education
ax = axes[0, 2]
surv_edu = df.groupby("Owner_Education")["Survived_3_Years"].mean().sort_values()
surv_edu.plot(kind="bar", ax=ax, color="#9b59b6", rot=0)
ax.set_title("Survival Rate by Education")
ax.set_ylabel("Survival Rate")

# 4. Startup capital distribution (log scale)
ax = axes[1, 0]
df["Startup_Capital_CFA"].apply(np.log10).hist(
    bins=40, ax=ax, color="#f39c12", edgecolor="white"
)
ax.set_title("Log10(Startup Capital) Distribution")
ax.set_xlabel("Log10(XAF)")

# 5. Power outage distribution
ax = axes[1, 1]
df["Power_Outage_Frequency"].clip(0, 25).hist(
    bins=26, ax=ax, color="#1abc9c", edgecolor="white"
)
ax.set_title("Power Outage Frequency (clipped at 25)")
ax.set_xlabel("Outages per month")

# 6. Financing source survival
ax = axes[1, 2]
fin_surv = (df.groupby("Financing_Source")["Survived_3_Years"]
            .mean().sort_values(ascending=True))
fin_surv.plot(kind="barh", ax=ax, color="#e67e22")
ax.set_title("Survival Rate by Financing Source")
ax.set_xlabel("Survival Rate")

plt.tight_layout()
chart_path = BASE_DIR / "eda_charts.png"
plt.savefig(chart_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Chart saved: {chart_path}")

# Numeric summary
print("\n── Numeric Summary ──")
print(df[["Startup_Capital_CFA", "Monthly_Profit_CFA", "Business_Age_Years",
          "Years_of_Experience", "Employees", "Power_Outage_Frequency"]].describe().round(1))


Chart saved: D:\bizness_APP\eda_charts.png

── Numeric Summary ──
       Startup_Capital_CFA  Monthly_Profit_CFA  Business_Age_Years  \
count              15000.0             15000.0             15000.0   
mean             1412829.7             17382.4                 3.5   
std              2287028.3             28983.9                 3.8   
min                 6396.0              5000.0                 0.1   
25%               187649.2              5000.0                 0.9   
50%               650092.5              6370.5                 2.2   
75%              1636815.5             17612.0                 4.6   
max             46268003.0           1060673.0                20.0   

       Years_of_Experience  Employees  Power_Outage_Frequency  
count              15000.0    15000.0                 15000.0  
mean                   4.4        6.9                     8.6  
std                    4.6       11.8                     4.1  
min                    0.0        1.0          

In [6]:
"""
CELL 6 — Feature Engineering
Create informative derived features. DRY: each transformation is a function.
"""

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add derived features that capture non-linear interactions
    not immediately visible to tree-based models.
    """
    df = df.copy()

    # --- Capital efficiency & scale ---
    df["log_startup_capital"]   = np.log1p(df["Startup_Capital_CFA"])
    df["log_monthly_profit"]    = np.log1p(df["Monthly_Profit_CFA"])
    df["capital_per_employee"]  = np.log1p(df["Startup_Capital_CFA"] / df["Employees"].clip(1))
    df["profit_on_capital"]     = df["Monthly_Profit_CFA"] / df["Startup_Capital_CFA"].clip(1)

    # --- Overhead pressure ---
    df["total_overhead_pct"]      = df["Energy_Cost_Percentage"] + df["Transport_Cost_Percentage"]
    df["infra_energy_pressure"]   = (1 - df["Region_Infrastructure_Index"]) * df["Energy_Cost_Percentage"]
    df["electricity_burden"]      = df["Power_Outage_Frequency"] * df["Energy_Cost_Percentage"] / 100.0

    # --- Owner maturity ---
    df["owner_experience_ratio"]  = df["Years_of_Experience"] / df["Owner_Age"].clip(18)
    df["age_experience_gap"]      = df["Owner_Age"] - df["Years_of_Experience"]

    # --- Business maturity ---
    df["log_business_age"]        = np.log1p(df["Business_Age_Years"])
    df["capital_to_age_ratio"]    = df["log_startup_capital"] / df["log_business_age"].clip(0.1)

    # --- Regional risk composite ---
    df["regional_risk_score"]     = (
        (1 - df["Region_Security_Index"]) * 0.4 +
        (1 - df["Region_Infrastructure_Index"]) * 0.3 +
        (1 - df["Region_Electricity_Index"]) * 0.3
    )

    # --- Competition pressure ---
    comp_map = {"Low": 0, "Medium": 1, "High": 2, "Very High": 3}
    df["competition_numeric"]     = df["Market_Competition"].map(comp_map).fillna(1)
    df["competition_x_outage"]    = df["competition_numeric"] * df["Power_Outage_Frequency"]

    # --- Formality index (0-3 scale: registration + records + business plan) ---
    df["formality_index"] = (
        (df["Registered_Formal"] == "Yes").astype(int) +
        (df["Formal_Financial_Records"] == "Yes").astype(int) +
        (df["Has_Business_Plan"] == "Yes").astype(int)
    )

    return df

df = engineer_features(df)
new_cols = ["log_startup_capital", "capital_per_employee", "total_overhead_pct",
            "electricity_burden", "regional_risk_score", "formality_index",
            "competition_numeric", "owner_experience_ratio"]
print(f"Engineered {len(new_cols)} new features (sample):")
print(df[new_cols].describe().round(3))


Engineered 8 new features (sample):
       log_startup_capital  capital_per_employee  total_overhead_pct  \
count            15000.000             15000.000           15000.000   
mean                13.265                12.110              23.632   
std                  1.421                 1.277               6.934   
min                  8.764                 7.646               5.000   
25%                 12.142                11.298              18.737   
50%                 13.385                11.989              23.600   
75%                 14.308                12.805              28.350   
max                 17.650                17.015              51.910   

       electricity_burden  regional_risk_score  formality_index  \
count           15000.000            15000.000        15000.000   
mean                1.163                0.260            1.391   
std                 0.749                0.105            0.912   
min                 0.000                0.142 

In [7]:
"""
CELL 7 — Encode categorical variables and define the feature lists
for survival model and profit model separately.
All LabelEncoders are stored for use during inference.
"""

# Categorical columns to encode
CATEGORICAL_COLS = [
    "Region", "Urban_Rural", "Sector", "Business_Type", "SME_Category_Cameroon",
    "Owner_Education", "Owner_Gender", "Access_to_Financing", "Financing_Source",
    "Registered_Formal", "Tax_Compliance_Level", "Has_Business_Plan",
    "Formal_Financial_Records", "Market_Competition", "Primary_Constraint",
    "Secondary_Constraint",
]

# Store encoders so inference can use the same mapping
label_encoders: dict = {}

df_encoded = df.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

# Drop ID and simulated/target-leaking columns
DROP_COLS = [
    "Business_ID", "Location",          # high cardinality, not useful
    "Survival_Probability_Simulated",   # data leak (used to generate target)
    "Risk_Level_Simulated",             # data leak
    "Monthly_Profit_CFA",              # excluded from survival; used as profit target
    "log_monthly_profit",              # same
    "Survived_3_Years",                # target
    "Business_Age_Years",              # kept separately for Cox PH time variable
]

# ── Survival feature list ───────────────────────────────────────────────────
SURVIVAL_FEATURES = [c for c in df_encoded.columns if c not in DROP_COLS]

# ── Profit feature list (trained on survivors only) ─────────────────────────
# Remove profit-related features to avoid leakage
PROFIT_DROP_EXTRA = ["profit_on_capital", "capital_to_age_ratio"]
PROFIT_FEATURES   = [c for c in SURVIVAL_FEATURES
                     if c not in PROFIT_DROP_EXTRA]

# Targets
SURVIVAL_TARGET = "Survived_3_Years"
PROFIT_TARGET   = "Monthly_Profit_CFA"   # raw; log-transform when training
COX_TIME_COL    = "Business_Age_Years"

print(f"Survival features : {len(SURVIVAL_FEATURES)}")
print(f"Profit features   : {len(PROFIT_FEATURES)}")
print("\nSample survival features:")
print(SURVIVAL_FEATURES[:10])


Survival features : 49
Profit features   : 47

Sample survival features:
['Region', 'Urban_Rural', 'Sector', 'Business_Type', 'SME_Category_Cameroon', 'Startup_Capital_CFA', 'Employees', 'Market_Competition', 'Years_of_Experience', 'Owner_Education']


In [8]:
"""
CELL 8 — Stratified train / validation / test split.
Stratify on survival label to preserve the ~17% minority in every split.
"""
X_all = df_encoded[SURVIVAL_FEATURES].values
y_all = df_encoded[SURVIVAL_TARGET].values

# 1. Carve out 15% test set (never touched during training)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_RATIO, random_state=SEED, stratify=y_all
)

# 2. From the remaining data, carve out validation set
val_fraction = VAL_RATIO / (1 - TEST_RATIO)   # adjust fraction for reduced pool
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_fraction, random_state=SEED, stratify=y_trainval
)

print("Split sizes:")
print(f"  Train       : {len(X_train):>6,}  |  survival rate: {y_train.mean():.1%}")
print(f"  Validation  : {len(X_val):>6,}  |  survival rate: {y_val.mean():.1%}")
print(f"  Test        : {len(X_test):>6,}  |  survival rate: {y_test.mean():.1%}")

# Class imbalance ratio (used for scale_pos_weight in LightGBM / XGBoost)
neg_count   = (y_train == 0).sum()
pos_count   = (y_train == 1).sum()
POS_WEIGHT  = neg_count / pos_count   # ~5x weight on minority (survived)
print(f"\nClass imbalance ratio (neg/pos): {POS_WEIGHT:.2f}")

# Profit data (survivors only in training set)
df_train = df_encoded.iloc[
    df_encoded.index.isin(
        df_encoded.sample(frac=1, random_state=SEED).index  # placeholder — will fix below
    )
]
# Correctly index train rows for profit model
train_idx, test_idx = train_test_split(
    np.arange(len(df_encoded)), test_size=TEST_RATIO, random_state=SEED,
    stratify=y_all
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=val_fraction, random_state=SEED,
    stratify=y_all[train_idx]
)

df_profit_train = df_encoded.iloc[train_idx][df_encoded.iloc[train_idx][SURVIVAL_TARGET] == 1]
df_profit_val   = df_encoded.iloc[val_idx][df_encoded.iloc[val_idx][SURVIVAL_TARGET] == 1]
df_profit_test  = df_encoded.iloc[test_idx][df_encoded.iloc[test_idx][SURVIVAL_TARGET] == 1]

print(f"\nProfit model rows (survivors only):")
print(f"  Train: {len(df_profit_train):,} | Val: {len(df_profit_val):,} | Test: {len(df_profit_test):,}")

X_profit_train = df_profit_train[PROFIT_FEATURES].values
X_profit_val   = df_profit_val[PROFIT_FEATURES].values
X_profit_test  = df_profit_test[PROFIT_FEATURES].values
y_profit_train = np.log1p(df_profit_train[PROFIT_TARGET].values)   # log-transform target
y_profit_val   = np.log1p(df_profit_val[PROFIT_TARGET].values)
y_profit_test  = np.log1p(df_profit_test[PROFIT_TARGET].values)


Split sizes:
  Train       : 10,500  |  survival rate: 17.7%
  Validation  :  2,250  |  survival rate: 17.7%
  Test        :  2,250  |  survival rate: 17.7%

Class imbalance ratio (neg/pos): 4.65

Profit model rows (survivors only):
  Train: 1,857 | Val: 398 | Test: 398


In [9]:
"""
CELL 9 — Train four baseline survival models and evaluate on validation set.
Models: LightGBM · CatBoost · Logistic Regression · Random Forest
Metric: ROC-AUC (primary), PR-AUC, Brier Score
"""

# ── Shared CV strategy ──────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)


def eval_model(model, X_val, y_val, name: str) -> dict:
    """Return a dict of key metrics for a fitted model."""
    proba = model.predict_proba(X_val)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    return {
        "Model"      : name,
        "ROC-AUC"    : round(roc_auc_score(y_val, proba), 4),
        "PR-AUC"     : round(average_precision_score(y_val, proba), 4),
        "Brier"      : round(brier_score_loss(y_val, proba), 4),
        "F1-Survived": round(f1_score(y_val, pred, pos_label=1, zero_division=0), 4),
    }


# ── 1. LightGBM ─────────────────────────────────────────────────────────────
lgb_base = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    scale_pos_weight=POS_WEIGHT,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
lgb_base.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)],
)

# ── 2. CatBoost ─────────────────────────────────────────────────────────────
cat_base = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=0,
    allow_writing_files=False,
)
cat_base.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# ── 3. Logistic Regression (needs scaled features) ──────────────────────────
scaler_lr = StandardScaler()
X_train_sc = scaler_lr.fit_transform(X_train)
X_val_sc   = scaler_lr.transform(X_val)
X_test_sc  = scaler_lr.transform(X_test)

lr_base = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs",
    C=0.5,
    random_state=SEED,
)
lr_base.fit(X_train_sc, y_train)

# ── 4. Random Forest ────────────────────────────────────────────────────────
rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED,
)
rf_base.fit(X_train, y_train)

# ── Comparison table ─────────────────────────────────────────────────────────
results = [
    eval_model(lgb_base, X_val, y_val, "LightGBM"),
    eval_model(cat_base, X_val, y_val, "CatBoost"),
    eval_model(lr_base,  X_val_sc, y_val, "Logistic Regression"),
    eval_model(rf_base,  X_val, y_val, "Random Forest"),
]
baseline_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
print("── Baseline Model Comparison (Validation Set) ──")
print(baseline_df.to_string(index=False))


── Baseline Model Comparison (Validation Set) ──
              Model  ROC-AUC  PR-AUC  Brier  F1-Survived
           CatBoost   0.7255  0.3785 0.2103       0.4172
Logistic Regression   0.7148  0.3686 0.2218       0.3985
           LightGBM   0.6860  0.3338 0.1425       0.0000
      Random Forest   0.6860  0.3364 0.1544       0.2921


In [10]:
"""
CELL 10 — Optuna hyperparameter tuning for LightGBM.
Uses stratified k-fold CV on the combined train+val set.
Objective: maximise ROC-AUC.
"""

X_trainval_arr = np.vstack([X_train, X_val])
y_trainval_arr = np.concatenate([y_train, y_val])


def lgb_objective(trial: optuna.Trial) -> float:
    """Optuna objective — returns mean CV ROC-AUC for a trial."""
    params = {
        "n_estimators"        : trial.suggest_int("n_estimators", 300, 1000),
        "learning_rate"       : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves"          : trial.suggest_int("num_leaves", 20, 150),
        "max_depth"           : trial.suggest_int("max_depth", 3, 10),
        "min_child_samples"   : trial.suggest_int("min_child_samples", 10, 100),
        "subsample"           : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree"    : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"           : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"          : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "scale_pos_weight"    : POS_WEIGHT,
        "random_state"        : SEED,
        "n_jobs"              : -1,
        "verbose"             : -1,
    }
    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(
        model, X_trainval_arr, y_trainval_arr,
        cv=skf, scoring="roc_auc", n_jobs=-1
    )
    return scores.mean()


print(f"Running {OPTUNA_TRIALS} Optuna trials for LightGBM...")
lgb_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_lgb_params = lgb_study.best_params
best_lgb_params.update({"scale_pos_weight": POS_WEIGHT, "random_state": SEED,
                          "n_jobs": -1, "verbose": -1})
print(f"\nBest LightGBM ROC-AUC (CV): {lgb_study.best_value:.4f}")
print("Best params:", best_lgb_params)

# Train tuned model on full train+val, evaluate on held-out test
lgb_tuned = lgb.LGBMClassifier(**best_lgb_params)
lgb_tuned.fit(X_trainval_arr, y_trainval_arr)
print("\nTest set metrics:")
print(eval_model(lgb_tuned, X_test, y_test, "LightGBM Tuned"))


Running 60 Optuna trials for LightGBM...


  0%|          | 0/60 [00:00<?, ?it/s]


Best LightGBM ROC-AUC (CV): 0.6980
Best params: {'n_estimators': 535, 'learning_rate': 0.015035253718568963, 'num_leaves': 70, 'max_depth': 3, 'min_child_samples': 33, 'subsample': 0.7678610923685274, 'colsample_bytree': 0.5510586145991498, 'reg_alpha': 9.293705934835252, 'reg_lambda': 4.048473751583619, 'scale_pos_weight': np.float64(4.654281098546042), 'random_state': 42, 'n_jobs': -1, 'verbose': -1}

Test set metrics:
{'Model': 'LightGBM Tuned', 'ROC-AUC': 0.6929, 'PR-AUC': 0.3461, 'Brier': 0.2133, 'F1-Survived': 0.3861}


In [11]:
"""
CELL 11 — Optuna hyperparameter tuning for CatBoost.
Same strategy as LightGBM cell — CV on train+val, report on test.
"""

def cat_objective(trial: optuna.Trial) -> float:
    """Optuna objective for CatBoost — returns mean CV ROC-AUC."""
    params = {
        "iterations"          : trial.suggest_int("iterations", 300, 1000),
        "learning_rate"       : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth"               : trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg"         : trial.suggest_float("l2_leaf_reg", 1.0, 30.0),
        "bagging_temperature" : trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "border_count"        : trial.suggest_int("border_count", 32, 255),
        "auto_class_weights"  : "Balanced",
        "random_seed"         : SEED,
        "verbose"             : 0,
        "allow_writing_files" : False,
    }
    model = CatBoostClassifier(**params)
    scores = cross_val_score(
        model, X_trainval_arr, y_trainval_arr,
        cv=skf, scoring="roc_auc", n_jobs=1   # CatBoost manages threads internally
    )
    return scores.mean()


print(f"Running {OPTUNA_TRIALS} Optuna trials for CatBoost...")
cat_study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
cat_study.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_cat_params = cat_study.best_params
best_cat_params.update({
    "auto_class_weights": "Balanced",
    "random_seed": SEED,
    "verbose": 0,
    "allow_writing_files": False,
})
print(f"\nBest CatBoost ROC-AUC (CV): {cat_study.best_value:.4f}")
print("Best params:", best_cat_params)

cat_tuned = CatBoostClassifier(**best_cat_params)
cat_tuned.fit(X_trainval_arr, y_trainval_arr)
print("\nTest set metrics:")
print(eval_model(cat_tuned, X_test, y_test, "CatBoost Tuned"))


Running 60 Optuna trials for CatBoost...


  0%|          | 0/60 [00:00<?, ?it/s]


Best CatBoost ROC-AUC (CV): 0.7025
Best params: {'iterations': 579, 'learning_rate': 0.014661168937006678, 'depth': 4, 'l2_leaf_reg': 27.263883936438138, 'bagging_temperature': 1.844478830709129, 'border_count': 189, 'auto_class_weights': 'Balanced', 'random_seed': 42, 'verbose': 0, 'allow_writing_files': False}

Test set metrics:
{'Model': 'CatBoost Tuned', 'ROC-AUC': 0.6994, 'PR-AUC': 0.3527, 'Brier': 0.2139, 'F1-Survived': 0.3908}


In [12]:
"""
CELL 12 — Probability calibration.
Tree models are often poorly calibrated (overconfident).
Platt scaling (sigmoid) and isotonic regression are both tested.
Best calibrated model is selected by Brier score.
"""

def calibrate(base_model, X_tr, y_tr, method: str = "sigmoid"):
    """Wrap a model with CalibratedClassifierCV using pre-fit base."""
    cal = CalibratedClassifierCV(estimator=base_model, method=method, cv="prefit")
    cal.fit(X_tr, y_tr)     # only the calibrator is trained here
    return cal


# Calibrate LightGBM and CatBoost (the two best baseline models)
lgb_cal_sigmoid  = calibrate(lgb_tuned,  X_val, y_val, method="sigmoid")
lgb_cal_isotonic = calibrate(lgb_tuned,  X_val, y_val, method="isotonic")
cat_cal_sigmoid  = calibrate(cat_tuned,  X_val, y_val, method="sigmoid")
cat_cal_isotonic = calibrate(cat_tuned,  X_val, y_val, method="isotonic")

cal_results = []
for name, model, X_e, y_e in [
    ("LightGBM Sigmoid",  lgb_cal_sigmoid,  X_test,    y_test),
    ("LightGBM Isotonic", lgb_cal_isotonic, X_test,    y_test),
    ("CatBoost Sigmoid",  cat_cal_sigmoid,  X_test,    y_test),
    ("CatBoost Isotonic", cat_cal_isotonic, X_test,    y_test),
]:
    metrics = eval_model(model, X_e, y_e, name)
    cal_results.append(metrics)

cal_df = pd.DataFrame(cal_results).sort_values("Brier")
print("── Calibration Comparison (Test Set) ──")
print(cal_df.to_string(index=False))

# Pick best by Brier score (lower = better calibration)
best_cal_name = cal_df.iloc[0]["Model"]
if "LightGBM" in best_cal_name:
    BEST_SURVIVAL_MODEL = lgb_cal_sigmoid if "Sigmoid" in best_cal_name else lgb_cal_isotonic
else:
    BEST_SURVIVAL_MODEL = cat_cal_sigmoid if "Sigmoid" in best_cal_name else cat_cal_isotonic

print(f"\n✅ Best calibrated survival model: {best_cal_name}")

# Calibration plot
fig, ax = plt.subplots(figsize=(7, 5))
for label, model, Xi, yi in [
    ("LightGBM Sigmoid",  lgb_cal_sigmoid,  X_test, y_test),
    ("CatBoost Sigmoid",  cat_cal_sigmoid,  X_test, y_test),
    ("LightGBM Uncal",    lgb_tuned,        X_test, y_test),
]:
    prob_true, prob_pred = calibration_curve(yi, model.predict_proba(Xi)[:, 1], n_bins=10)
    ax.plot(prob_pred, prob_true, marker="o", label=label)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curves (Test Set)")
ax.legend(fontsize=9)
plt.tight_layout()
cal_path = BASE_DIR / "calibration_curves.png"
plt.savefig(cal_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"Chart saved: {cal_path}")


── Calibration Comparison (Test Set) ──
            Model  ROC-AUC  PR-AUC  Brier  F1-Survived
 CatBoost Sigmoid   0.6994  0.3527 0.1344       0.2127
CatBoost Isotonic   0.7002  0.3399 0.1344       0.1752
 LightGBM Sigmoid   0.6929  0.3461 0.1361       0.2099
LightGBM Isotonic   0.6906  0.3255 0.1374       0.2116

✅ Best calibrated survival model: CatBoost Sigmoid
Chart saved: D:\bizness_APP\calibration_curves.png


In [13]:
"""
CELL 13 — Cox Proportional Hazards (CoxPH) survival model.
CoxPH models the hazard rate over time, giving us a proper time-to-event
perspective rather than just a binary snapshot.
Time variable: Business_Age_Years
Event variable: Survived_3_Years (1 = event occurred = business survived 3yrs)
"""

# Select a compact feature set for CoxPH (must handle multicollinearity)
COX_FEATURES = [
    "Business_Age_Years",           # time variable
    "Survived_3_Years",             # event indicator
    "log_startup_capital",
    "Employees",
    "Years_of_Experience",
    "Owner_Education",              # encoded integer
    "Access_to_Financing",          # encoded integer
    "Financing_Source",             # encoded integer
    "Registered_Formal",            # encoded integer
    "Has_Business_Plan",            # encoded integer
    "Formal_Financial_Records",     # encoded integer
    "formality_index",
    "regional_risk_score",
    "total_overhead_pct",
    "Power_Outage_Frequency",
    "Electricity_Access_Score",
    "Sector_Risk_Baseline",
    "competition_numeric",
]

df_cox = df_encoded.iloc[train_idx][COX_FEATURES].copy()

# Fit CoxPH
cox_model = CoxPHFitter(penalizer=0.10)  # L2 penalty for stability
cox_model.fit(
    df_cox,
    duration_col="Business_Age_Years",
    event_col="Survived_3_Years",
    show_progress=False,
)

cox_model.print_summary(decimals=3)

# Concordance index on test set
df_cox_test = df_encoded.iloc[test_idx][COX_FEATURES].copy()
c_index = concordance_index(
    df_cox_test["Business_Age_Years"],
    -cox_model.predict_partial_hazard(df_cox_test),
    df_cox_test["Survived_3_Years"],
)
print(f"\nCox PH C-index (test): {c_index:.4f}  (0.5=random, 1.0=perfect)")


<lifelines.CoxPHFitter: fitted with 10500 total observations, 8643 right-censored observations>
             duration col = 'Business_Age_Years'
                event col = 'Survived_3_Years'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 10500
number of events observed = 1857
   partial log-likelihood = -14818.688
         time fit was run = 2026-05-08 21:56:40 UTC

---
                           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                  
log_startup_capital      -0.028     0.973     0.015          -0.056           0.001               0.945               1.001
Employees                -0.008     0.992     0.002          -0.011          -0.005               0.989               0.995
Years_of_Experience       0.007     1.007     0.004          -0.000           0.014               1.000               1.015
Owner_Education           0.196     1.217     0.027           0.144           0.249               1.155               1.283
Access_to_Financing       0.254     1.289     0.042           0.171           0.337               1.186               1.400
Financing_Source         -0.040     0.961     0.019          -0.078          -0.002               0.925               0.998
Registered_Formal         0.082     1.085     0.044          -0.003           0.167               0.997               1.182
Has_Business_Plan         0.038     1.038     0.044          -0.049           0.124               0.952               1.133
Formal_Financial_Records  0.107     1.113     0.042           0.025           0.190               1.025               1.209
formality_index           0.066     1.068     0.027           0.013           0.120               1.013               1.127
regional_risk_score      -1.517     0.219     0.205          -1.919          -1.115               0.147               0.328
total_overhead_pct       -0.004     0.996     0.003          -0.009           0.001               0.991               1.001
Power_Outage_Frequency   -0.007     0.993     0.005          -0.017           0.002               0.983               1.002
Electricity_Access_Score  0.121     1.129     0.020           0.082           0.161               1.085               1.175
Sector_Risk_Baseline     -0.320     0.726     0.423          -1.148           0.509               0.317               1.663
competition_numeric      -0.021     0.979     0.021          -0.063           0.020               0.939               1.021

                          cmp to      z       p  -log2(p)
covariate                                                
log_startup_capital        0.000 -1.905   0.057     4.139
Employees                  0.000 -5.065 <0.0005    21.221
Years_of_Experience        0.000  1.884   0.060     4.070
Owner_Education            0.000  7.315 <0.0005    41.819
Access_to_Financing        0.000  6.004 <0.0005    28.956
Financing_Source           0.000 -2.079   0.038     4.734
Registered_Formal          0.000  1.884   0.060     4.069
Has_Business_Plan          0.000  0.850   0.395     1.339
Formal_Financial_Records   0.000  2.545   0.011     6.517
formality_index            0.000  2.424   0.015     6.024
regional_risk_score        0.000 -7.396 <0.0005    42.694
total_overhead_pct         0.000 -1.518   0.129     2.954
Power_Outage_Frequency     0.000 -1.472   0.141     2.827
Electricity_Access_Score   0.000  5.993 <0.0005    28.859
Sector_Risk_Baseline       0.000 -0.757   0.449     1.154
competition_numeric        0.000 -0.993   0.321     1.640
---
Concordance = 0.647
Partial AIC = 29669.376
log-likelihood ratio test = 354.073 on 16 df
-log2(p) of ll-ratio test = 215.376


Cox PH C-index (test): 0.6208  (0.5=random, 1.0=perfect)


In [14]:
"""
CELL 14 — Ensemble: average probabilities from the two best calibrated models
plus the CoxPH partial hazard (normalised to [0,1]).
Simple averaging avoids overfitting to the validation set.
"""

def sigmoid(x):
    """Numerically stable sigmoid."""
    return 1 / (1 + np.exp(-x))


def ensemble_predict(X_arr, X_arr_for_cox_df) -> np.ndarray:
    """
    Average predicted survival probabilities from:
      - LightGBM (calibrated)
      - CatBoost (calibrated)
      - CoxPH (partial hazard converted to probability)
    """
    # Calibrated tree model probabilities
    lgb_proba = lgb_cal_sigmoid.predict_proba(X_arr)[:, 1]
    cat_proba = cat_cal_sigmoid.predict_proba(X_arr)[:, 1]

    # CoxPH: lower partial hazard = better survival → convert to [0,1]
    cox_hazard = cox_model.predict_partial_hazard(X_arr_for_cox_df).values
    cox_proba  = 1.0 - (cox_hazard - cox_hazard.min()) / (cox_hazard.max() - cox_hazard.min() + 1e-9)

    # Weighted average: tree models weighted higher (better calibration)
    ensemble_proba = 0.40 * lgb_proba + 0.40 * cat_proba + 0.20 * cox_proba
    return ensemble_proba


df_test_encoded = df_encoded.iloc[test_idx]

ensemble_proba = ensemble_predict(X_test, df_test_encoded[COX_FEATURES])
ensemble_pred  = (ensemble_proba >= 0.50).astype(int)

ens_roc    = roc_auc_score(y_test, ensemble_proba)
ens_prauc  = average_precision_score(y_test, ensemble_proba)
ens_brier  = brier_score_loss(y_test, ensemble_proba)
ens_f1     = f1_score(y_test, ensemble_pred, pos_label=1, zero_division=0)

print("── Ensemble Model — Test Set ──")
print(f"ROC-AUC  : {ens_roc:.4f}")
print(f"PR-AUC   : {ens_prauc:.4f}")
print(f"Brier    : {ens_brier:.4f}")
print(f"F1-Surv  : {ens_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, ensemble_pred, target_names=["Failed", "Survived"]))


── Ensemble Model — Test Set ──
ROC-AUC  : 0.6867
PR-AUC   : 0.3431
Brier    : 0.1441
F1-Surv  : 0.2008

Classification Report:
              precision    recall  f1-score   support

      Failed       0.84      0.98      0.90      1852
    Survived       0.54      0.12      0.20       398

    accuracy                           0.83      2250
   macro avg       0.69      0.55      0.55      2250
weighted avg       0.79      0.83      0.78      2250



In [15]:
"""
CELL 15 — Profit regression (forward-looking projected monthly profit).
Trained ONLY on businesses that survived 3 years (Survived_3_Years == 1).
Target: log1p(Monthly_Profit_CFA) — log-transform to handle right skew.
Models: LightGBM · XGBoost · Ridge · CatBoost  →  best by RMSE on val set.
"""

def rmse(y_true, y_pred):
    """Root Mean Squared Error helper."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def r2(y_true, y_pred):
    """R-squared helper."""
    return r2_score(y_true, y_pred)


# ── 1. LightGBM Regressor ──────────────────────────────────────────────────
lgb_reg = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_reg.fit(
    X_profit_train, y_profit_train,
    eval_set=[(X_profit_val, y_profit_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

# ── 2. XGBoost Regressor ───────────────────────────────────────────────────
xgb_reg = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbosity=0,
)
xgb_reg.fit(
    X_profit_train, y_profit_train,
    eval_set=[(X_profit_val, y_profit_val)],
    verbose=False,
)

# ── 3. Ridge Regression (scaled) ───────────────────────────────────────────
scaler_profit = StandardScaler()
X_pr_tr_sc = scaler_profit.fit_transform(X_profit_train)
X_pr_val_sc = scaler_profit.transform(X_profit_val)
X_pr_test_sc = scaler_profit.transform(X_profit_test)

ridge_reg = Ridge(alpha=10.0, random_state=SEED)
ridge_reg.fit(X_pr_tr_sc, y_profit_train)

# ── 4. CatBoost Regressor ──────────────────────────────────────────────────
cat_reg = CatBoostRegressor(
    iterations=500, learning_rate=0.05, depth=6,
    random_seed=SEED, verbose=0, allow_writing_files=False
)
cat_reg.fit(X_profit_train, y_profit_train,
            eval_set=(X_profit_val, y_profit_val),
            early_stopping_rounds=50)

# ── Optuna tuning for LightGBM regressor ────────────────────────────────────
def lgb_reg_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 300, 800),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves"       : trial.suggest_int("num_leaves", 20, 120),
        "max_depth"        : trial.suggest_int("max_depth", 3, 9),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "random_state"     : SEED, "n_jobs": -1, "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    # Manual CV using sklearn StratifiedKFold is not valid for regression;
    # use a simple val-set RMSE instead (efficient for regression)
    model.fit(X_profit_train, y_profit_train)
    val_pred = model.predict(X_profit_val)
    return rmse(y_profit_val, val_pred)   # minimise


print(f"Tuning LightGBM regressor ({OPTUNA_TRIALS} trials)...")
lgb_reg_study = optuna.create_study(direction="minimize",
                                     sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_reg_study.optimize(lgb_reg_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_lgb_reg_params = lgb_reg_study.best_params
best_lgb_reg_params.update({"random_state": SEED, "n_jobs": -1, "verbose": -1})
lgb_reg_tuned = lgb.LGBMRegressor(**best_lgb_reg_params)
lgb_reg_tuned.fit(
    np.vstack([X_profit_train, X_profit_val]),
    np.concatenate([y_profit_train, y_profit_val])
)

# ── Comparison ──────────────────────────────────────────────────────────────
profit_results = []
for name, model, Xp in [
    ("LightGBM Tuned", lgb_reg_tuned, X_profit_test),
    ("XGBoost",        xgb_reg,       X_profit_test),
    ("CatBoost",       cat_reg,       X_profit_test),
    ("Ridge",          ridge_reg,     X_pr_test_sc),
]:
    pred_log = model.predict(Xp)
    pred_raw = np.expm1(pred_log)
    true_raw = np.expm1(y_profit_test)
    profit_results.append({
        "Model"    : name,
        "RMSE-log" : round(rmse(y_profit_test, pred_log), 4),
        "R2-log"   : round(r2(y_profit_test, pred_log), 4),
        "MAE-XAF"  : round(np.mean(np.abs(true_raw - pred_raw)), 0),
    })

profit_df = pd.DataFrame(profit_results).sort_values("RMSE-log")
print("── Profit Model Comparison (Test Set) ──")
print(profit_df.to_string(index=False))

# Best profit model
BEST_PROFIT_MODEL  = lgb_reg_tuned   # typically LightGBM wins
PROFIT_SCALER      = None             # not needed for LightGBM
print("\n✅ Best profit model: LightGBM Tuned")


Tuning LightGBM regressor (60 trials)...


  0%|          | 0/60 [00:00<?, ?it/s]

── Profit Model Comparison (Test Set) ──
         Model  RMSE-log  R2-log  MAE-XAF
LightGBM Tuned    0.3097  0.8963   5982.0
      CatBoost    0.3119  0.8948   5788.0
       XGBoost    0.3317  0.8810   6349.0
         Ridge    0.4327  0.7975   8386.0

✅ Best profit model: LightGBM Tuned


In [16]:
"""
CELL 16 — SHAP analysis for survival model (LightGBM tuned).
Answers: which features push a business toward survival or failure?
"""

print("Computing SHAP values for survival model (sample of 1000)...")

# Use a random sample to keep computation fast
shap_sample_size = min(1000, len(X_test))
shap_idx = np.random.choice(len(X_test), shap_sample_size, replace=False)
X_shap = X_test[shap_idx]

# TreeExplainer is the correct explainer for LightGBM
explainer_survival = shap.TreeExplainer(lgb_tuned)
shap_values_surv   = explainer_survival.shap_values(X_shap)

# shap_values returns a list [class0, class1] for classifiers
if isinstance(shap_values_surv, list):
    shap_vals_pos = shap_values_surv[1]   # class 1 = survived
else:
    shap_vals_pos = shap_values_surv

# Bar plot — global feature importance
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_vals_pos, X_shap,
    feature_names=SURVIVAL_FEATURES,
    plot_type="bar",
    show=False, max_display=20,
)
plt.title("SHAP Feature Importance — Survival Model (Top 20)")
plt.tight_layout()
shap_path = BASE_DIR / "shap_survival_importance.png"
plt.savefig(shap_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"SHAP bar chart saved: {shap_path}")

# Beeswarm plot — direction and magnitude
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals_pos, X_shap,
    feature_names=SURVIVAL_FEATURES,
    show=False, max_display=20,
)
plt.title("SHAP Beeswarm — Survival Model")
plt.tight_layout()
shap_bee_path = BASE_DIR / "shap_survival_beeswarm.png"
plt.savefig(shap_bee_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"SHAP beeswarm saved: {shap_bee_path}")


Computing SHAP values for survival model (sample of 1000)...
SHAP bar chart saved: D:\bizness_APP\shap_survival_importance.png
SHAP beeswarm saved: D:\bizness_APP\shap_survival_beeswarm.png


In [17]:
"""
CELL 17 — SHAP analysis for profit model (LightGBM regressor tuned).
Answers: what drives projected monthly profit for surviving businesses?
"""

print("Computing SHAP values for profit model...")

shap_profit_size = min(500, len(X_profit_test))
shap_profit_idx  = np.random.choice(len(X_profit_test), shap_profit_size, replace=False)
X_shap_profit    = X_profit_test[shap_profit_idx]

explainer_profit  = shap.TreeExplainer(lgb_reg_tuned)
shap_values_profit = explainer_profit.shap_values(X_shap_profit)

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_profit, X_shap_profit,
    feature_names=PROFIT_FEATURES,
    plot_type="bar",
    show=False, max_display=20,
)
plt.title("SHAP Feature Importance — Profit Model (Top 20)")
plt.tight_layout()
shap_profit_path = BASE_DIR / "shap_profit_importance.png"
plt.savefig(shap_profit_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"SHAP profit chart saved: {shap_profit_path}")


Computing SHAP values for profit model...
SHAP profit chart saved: D:\bizness_APP\shap_profit_importance.png


In [18]:
"""
CELL 18 — Final evaluation: compare all models on the held-out test set.
This is the honest, never-seen-before evaluation.
"""

final_results = []

# All survival classifiers on test set
for name, model, Xi in [
    ("LightGBM Baseline",   lgb_base,         X_test),
    ("CatBoost Baseline",   cat_base,          X_test),
    ("LR Baseline",         lr_base,           X_test_sc),
    ("RF Baseline",         rf_base,           X_test),
    ("LightGBM Tuned",      lgb_tuned,         X_test),
    ("CatBoost Tuned",      cat_tuned,         X_test),
    ("LightGBM Cal (best)", BEST_SURVIVAL_MODEL, X_test),
    ("Cat Cal Sigmoid",     cat_cal_sigmoid,   X_test),
]:
    m = eval_model(model, Xi, y_test, name)
    final_results.append(m)

# Ensemble row
final_results.append({
    "Model"      : "Ensemble (LGB+CAT+COX)",
    "ROC-AUC"    : round(roc_auc_score(y_test, ensemble_proba), 4),
    "PR-AUC"     : round(average_precision_score(y_test, ensemble_proba), 4),
    "Brier"      : round(brier_score_loss(y_test, ensemble_proba), 4),
    "F1-Survived": round(f1_score(y_test, ensemble_pred, pos_label=1, zero_division=0), 4),
})

final_df = pd.DataFrame(final_results).sort_values("ROC-AUC", ascending=False)
print("═" * 70)
print("FINAL MODEL EVALUATION — Test Set (never seen during training)")
print("═" * 70)
print(final_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(final_df["Model"], final_df["ROC-AUC"], color="#3498db", edgecolor="white")
ax.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="Random (0.50)")
ax.set_xlabel("ROC-AUC")
ax.set_title("Model Comparison — ROC-AUC on Test Set")
ax.legend()
plt.tight_layout()
comparison_path = BASE_DIR / "model_comparison.png"
plt.savefig(comparison_path, dpi=100, bbox_inches="tight")
plt.show()
print(f"\nComparison chart saved: {comparison_path}")

# Cox PH summary
print(f"\nCox PH Concordance Index: {c_index:.4f}")
print("\n✅ Training complete. Proceed to Cell 19 to save models.")


══════════════════════════════════════════════════════════════════════
FINAL MODEL EVALUATION — Test Set (never seen during training)
══════════════════════════════════════════════════════════════════════
                 Model  ROC-AUC  PR-AUC  Brier  F1-Survived
        CatBoost Tuned   0.6994  0.3527 0.2139       0.3908
   LightGBM Cal (best)   0.6994  0.3527 0.1344       0.2127
       Cat Cal Sigmoid   0.6994  0.3527 0.1344       0.2127
           LR Baseline   0.6941  0.3412 0.2198       0.3770
        LightGBM Tuned   0.6929  0.3461 0.2133       0.3861
     CatBoost Baseline   0.6884  0.3390 0.2108       0.3842
Ensemble (LGB+CAT+COX)   0.6867  0.3431 0.1441       0.2008
           RF Baseline   0.6698  0.3165 0.1554       0.2857
     LightGBM Baseline   0.6570  0.3043 0.1435       0.0000

Comparison chart saved: D:\bizness_APP\model_comparison.png

Cox PH Concordance Index: 0.6208

✅ Training complete. Proceed to Cell 19 to save models.


In [19]:
"""
CELL 19 — Save all V3 models and artefacts to backend/models/.
These files are loaded by ml_service.py during inference.
"""

def save_pkl(obj, filename: str) -> None:
    """Pickle an object to MODELS_DIR and print confirmation."""
    path = MODELS_DIR / filename
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    size_kb = os.path.getsize(path) / 1024
    print(f"  ✅ Saved {filename} ({size_kb:.1f} KB)")


print(f"Saving V3 models to: {MODELS_DIR}\n")

# ── Survival models ─────────────────────────────────────────────────────────
save_pkl(BEST_SURVIVAL_MODEL,  "sme_survival_model_v3.pkl")       # primary (calibrated)
save_pkl(lgb_tuned,            "sme_survival_lgb_v3.pkl")          # raw LightGBM
save_pkl(cat_tuned,            "sme_survival_cat_v3.pkl")          # raw CatBoost
save_pkl(cox_model,            "sme_cox_ph_v3.pkl")               # Cox PH

# ── Profit model ─────────────────────────────────────────────────────────────
save_pkl(BEST_PROFIT_MODEL,   "sme_profit_model_v3.pkl")           # primary LightGBM

# ── Feature lists (inference must use the same features) ───────────────────
save_pkl(SURVIVAL_FEATURES,   "survival_feature_list_v3.pkl")
save_pkl(PROFIT_FEATURES,     "profit_feature_list_v3.pkl")
save_pkl(COX_FEATURES,        "cox_feature_list_v3.pkl")

# ── Label encoders (inference encodes categoricals the same way) ────────────
save_pkl(label_encoders,      "label_encoders_v3.pkl")

# ── Scalers (for LR baseline — keep for potential future use) ───────────────
save_pkl(scaler_lr,           "scaler_lr_v3.pkl")

print("\n── V3 Model Inventory ──")
for f in sorted(MODELS_DIR.glob("*_v3.pkl")):
    kb = os.path.getsize(f) / 1024
    print(f"  {f.name:<45}  {kb:>8.1f} KB")

print("\n🎉 All V3 models saved. Update ml_service.py to load from backend/models/.")
print("Next steps:")
print("  Phase 2 — Update Pydantic schemas with new features")
print("  Phase 4 — Swap Gemini → Groq in llm_service.py")
print("  Phase 4 — Fix Business_Age_Years=0 bug in ml_service.py")


Saving V3 models to: D:\bizness_APP\backend\models

  ✅ Saved sme_survival_model_v3.pkl (198.1 KB)
  ✅ Saved sme_survival_lgb_v3.pkl (516.3 KB)
  ✅ Saved sme_survival_cat_v3.pkl (197.6 KB)
  ✅ Saved sme_cox_ph_v3.pkl (810.8 KB)
  ✅ Saved sme_profit_model_v3.pkl (491.8 KB)
  ✅ Saved survival_feature_list_v3.pkl (1.0 KB)
  ✅ Saved profit_feature_list_v3.pkl (1.0 KB)
  ✅ Saved cox_feature_list_v3.pkl (0.4 KB)
  ✅ Saved label_encoders_v3.pkl (2.3 KB)
  ✅ Saved scaler_lr_v3.pkl (1.6 KB)

── V3 Model Inventory ──
  cox_feature_list_v3.pkl                             0.4 KB
  label_encoders_v3.pkl                               2.3 KB
  profit_feature_list_v3.pkl                          1.0 KB
  scaler_lr_v3.pkl                                    1.6 KB
  sme_cox_ph_v3.pkl                                 810.8 KB
  sme_profit_model_v3.pkl                           491.8 KB
  sme_survival_cat_v3.pkl                           197.6 KB
  sme_survival_lgb_v3.pkl                           516.3 KB